# Data preprocessing

This public notebook starts from neutral intermediate station records and derives the processed daily, monthly, and annual datasets used by the climate-index workflow. It does not read the original source exports directly.


In [1]:
from pathlib import Path

import pandas as pd


In [ ]:
def get_project_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == "scripts" else cwd


base_dir = get_project_root()
data_raw_dir = base_dir / "data_raw"
data_intermediate_dir = base_dir / "data_intermediate"
data_intermediate_dir.mkdir(parents=True, exist_ok=True)

intermediate_files = {
    "daily": data_intermediate_dir / "century_daily_station_records.csv",
    "monthly": data_intermediate_dir / "century_monthly_station_records.csv",
}

output_files = {
    "daily": data_raw_dir / "updated_df_daily.csv",
    "monthly": data_raw_dir / "updated_df_monthy.csv",
    "yearly": data_raw_dir / "updated_df_yearly.csv",
}

missing = [str(path) for path in intermediate_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing intermediate preprocessing input(s) "
        + ", ".join(missing)
    )


## Load intermediate station records


In [3]:
df_daily = pd.read_csv(intermediate_files["daily"])
df_monthly = pd.read_csv(intermediate_files["monthly"])


## Water-year summaries

Monthly records are clipped to water years 1926-2025. SPEI and SPI columns are limited to the conventional [-3, 3] range before annual aggregation.


In [4]:
def compute_water_year(date_series: pd.Series) -> pd.Series:
    dates = pd.to_datetime(date_series)
    return dates.dt.year + (dates.dt.month >= 10).astype(int)


def clip_spei_spi(df: pd.DataFrame, lower_bound: float = -3.0, upper_bound: float = 3.0) -> pd.DataFrame:
    clipped = df.copy()
    spei_spi_columns = [
        col for col in clipped.columns
        if "spei" in col.lower() or "spi" in col.lower()
    ]
    clipped[spei_spi_columns] = clipped[spei_spi_columns].clip(
        lower=lower_bound,
        upper=upper_bound,
    )
    return clipped


df_monthly["DATE"] = pd.to_datetime(df_monthly[["YEAR", "MONTH"]].assign(DAY=1))
df_monthly["W_YEAR"] = compute_water_year(df_monthly["DATE"])
df_monthly = df_monthly[
    (df_monthly["DATE"] >= "1925-10-01")
    & (df_monthly["DATE"] <= "2025-09-30")
].copy()

df_daily["DATE"] = pd.to_datetime(df_daily["DATE"])
df_daily["W_YEAR"] = compute_water_year(df_daily["DATE"])

df_monthly_updated = clip_spei_spi(df_monthly)

df_yearly = (
    df_monthly_updated
    .groupby(["ID", "W_YEAR"], as_index=False)
    .agg({
        "PRCP": "sum",
        "TMAX": "mean",
        "TMIN": "mean",
        "PET_thornwaite": "sum",
    })
)
df_yearly["AI"] = df_yearly["PRCP"] / df_yearly["PET_thornwaite"]
df_yearly = df_yearly.drop(columns="PET_thornwaite")
df_monthly_updated = df_monthly_updated.drop(columns="PET_thornwaite", errors="ignore")


## Temperature spell indices

WSDI and CSDI are calculated using 1966-1995 as the percentile reference period and six or more consecutive threshold-exceedance days as the spell criterion.


In [5]:
def _spell_index_by_station(
    station_df: pd.DataFrame,
    value_col: str,
    percentile: float,
    threshold_col: str,
    spell_flag_col: str,
    group_col: str,
    output_col: str,
    comparison: str,
) -> pd.DataFrame:
    station_df = station_df.copy().sort_values("DATE")
    base_period = station_df[(station_df["W_YEAR"] >= 1966) & (station_df["W_YEAR"] <= 1995)]

    percentiles = (
        base_period
        .groupby(["MONTH", "DAY"])[value_col]
        .apply(lambda x: x.rolling(window=5, center=True).quantile(percentile).dropna())
        .reset_index(name=threshold_col)
        .groupby(["MONTH", "DAY"], as_index=False)[threshold_col]
        .mean()
    )

    station_df = station_df.merge(percentiles, how="left", on=["MONTH", "DAY"])
    if comparison == "greater":
        station_df[spell_flag_col] = station_df[value_col] > station_df[threshold_col]
    elif comparison == "less":
        station_df[spell_flag_col] = station_df[value_col] < station_df[threshold_col]
    else:
        raise ValueError("comparison must be 'greater' or 'less'")

    station_df[group_col] = (station_df[spell_flag_col] != station_df[spell_flag_col].shift()).cumsum()
    spell_periods = station_df[station_df[spell_flag_col]]

    spell_counts = (
        spell_periods
        .groupby(["W_YEAR", group_col])
        .filter(lambda x: len(x) >= 6)
        .groupby("W_YEAR")["DAY"]
        .count()
        .reset_index(name=output_col)
    )

    all_years = pd.DataFrame({"W_YEAR": station_df["W_YEAR"].dropna().unique()})
    return all_years.merge(spell_counts, on="W_YEAR", how="left").fillna({output_col: 0})


def wsdi_agg(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for station_id, station_df in df.groupby("ID"):
        result = _spell_index_by_station(
            station_df,
            value_col="TMAX",
            percentile=0.9,
            threshold_col="TX90",
            spell_flag_col="is_warm_spell",
            group_col="warm_spell_group",
            output_col="WSDI",
            comparison="greater",
        )
        result["ID"] = station_id
        records.append(result)
    return pd.concat(records, ignore_index=True)


def csdi_agg(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for station_id, station_df in df.groupby("ID"):
        result = _spell_index_by_station(
            station_df,
            value_col="TMIN",
            percentile=0.1,
            threshold_col="TN10",
            spell_flag_col="is_cold_spell",
            group_col="cold_spell_group",
            output_col="CSDI",
            comparison="less",
        )
        result["ID"] = station_id
        records.append(result)
    return pd.concat(records, ignore_index=True)


wsdi = wsdi_agg(df_daily)
csdi = csdi_agg(df_daily)

df_wsdi_csdi = wsdi[["ID", "W_YEAR", "WSDI"]].merge(
    csdi[["ID", "W_YEAR", "CSDI"]],
    on=["ID", "W_YEAR"],
    how="left",
)


## Precipitation concentration and intensity indices

SDII is calculated from wet days with precipitation at least 1 mm. PCI is calculated from monthly precipitation concentration within each water year.


In [6]:
def calculate_sdii(df: pd.DataFrame) -> pd.DataFrame:
    wet_days = df[df["PRCP"] >= 1].copy()
    sdii = (
        wet_days
        .groupby(["ID", "W_YEAR"], as_index=False)
        .agg(total_precip=("PRCP", "sum"), wet_days=("PRCP", "count"))
    )
    sdii["SDII"] = sdii["total_precip"] / sdii["wet_days"]
    return sdii[["ID", "W_YEAR", "SDII"]]


def calculate_pci(df: pd.DataFrame) -> pd.DataFrame:
    monthly_precip = (
        df
        .groupby(["ID", "W_YEAR", "MONTH"], as_index=False)
        .agg(monthly_precip=("PRCP", "sum"))
    )
    monthly_precip["monthly_precip_sq"] = monthly_precip["monthly_precip"] ** 2
    annual = (
        monthly_precip
        .groupby(["ID", "W_YEAR"], as_index=False)
        .agg(annual_precip=("monthly_precip", "sum"), annual_precip_sq=("monthly_precip_sq", "sum"))
    )
    annual["PCI"] = annual["annual_precip_sq"] / (annual["annual_precip"] ** 2) * 100
    return annual[["ID", "W_YEAR", "PCI"]]


df_sdii = calculate_sdii(df_daily)
df_pci = calculate_pci(df_monthly_updated)

df_sdii_pci = df_sdii.merge(df_pci, on=["ID", "W_YEAR"], how="outer")


## Assemble final preprocessing outputs

The final dataframes are `df_daily_updated`, `df_monthly_updated`, and `df_yearly_updated`. Export commands remain commented so this notebook can be inspected without overwriting existing workflow data.


In [7]:
df_yearly_updated = (
    df_yearly
    .merge(df_wsdi_csdi, on=["ID", "W_YEAR"], how="left")
    .merge(df_sdii_pci, on=["ID", "W_YEAR"], how="left")
)
df_daily_updated = df_daily.copy()

summary = pd.DataFrame({
    "dataset": ["daily", "monthly", "yearly"],
    "rows": [len(df_daily_updated), len(df_monthly_updated), len(df_yearly_updated)],
    "stations": [
        df_daily_updated["ID"].nunique(),
        df_monthly_updated["ID"].nunique(),
        df_yearly_updated["ID"].nunique(),
    ],
    "first_water_year": [
        df_daily_updated["W_YEAR"].min(),
        df_monthly_updated["W_YEAR"].min(),
        df_yearly_updated["W_YEAR"].min(),
    ],
    "last_water_year": [
        df_daily_updated["W_YEAR"].max(),
        df_monthly_updated["W_YEAR"].max(),
        df_yearly_updated["W_YEAR"].max(),
    ],
})
summary


,dataset,rows,stations,first_water_year,last_water_year
0,daily,32151928,863,1924,2026
1,monthly,1035600,863,1926,2025
2,yearly,86300,863,1926,2025


## Save processed workflow datasets


In [8]:
df_daily_updated.to_csv(output_files["daily"], index=False)
df_monthly_updated.to_csv(output_files["monthly"], index=False)
df_yearly_updated.to_csv(output_files["yearly"], index=False)

pd.DataFrame({
    "dataset": ["daily", "monthly", "yearly"],
    "rows": [len(df_daily_updated), len(df_monthly_updated), len(df_yearly_updated)],
    "stations": [
        df_daily_updated["ID"].nunique(),
        df_monthly_updated["ID"].nunique(),
        df_yearly_updated["ID"].nunique(),
    ],
    "output_file": [str(output_files["daily"]), str(output_files["monthly"]), str(output_files["yearly"])],
})


,dataset,rows,stations,output_file
0,daily,32151928,863,C:\Users\adeba\OneDrive\Documents\Datascience\...
1,monthly,1035600,863,C:\Users\adeba\OneDrive\Documents\Datascience\...
2,yearly,86300,863,C:\Users\adeba\OneDrive\Documents\Datascience\...
